In [1]:
!pip install netCDF4 xarray cartopy networkx scipy scikit-learn


  Using cached netcdf4-1.7.4-cp311-abi3-macosx_14_0_arm64.whl.metadata (2.1 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 4.2 MB/s  0:00:02 eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached cftime-1.6.5-cp314-cp314-macosx_11_0_arm64.whl.metadata (8.7 kB)
  Using cached certifi-2026.6.17-py3-none-any.whl.metadata (2.5 kB)
  Using cached numpy-2.5.1-cp314-cp314-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached pandas-3.0.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached matplotlib-3.11.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (80 kB)
  Using cached narwhals-2.23.0-py3-none-any.whl.metadata (15 kB)
  Using cached contourpy-1.3.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp314-cp314-macosx_10_15_universal2.whl.metadata (118 kB)
  Us

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')
# from os import path as path

# in_dir = '/content/drive/MyDrive/Data'
# drive_dir = '/content/drive/MyDrive'
# l2_dir = '/content/drive/MyDrive/Masks/L2'
# l1_dir = '/content/drive/MyDrive/Masks/L1'

import xarray as xr
import pandas as pd
import pprint as pp
import math
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.dates as mdates
import cartopy.crs as ccrs
import matplotlib.colors as mcolors
import cartopy.feature as cfeature
import scipy.sparse as sp
import networkx as nx
import gc

In [3]:
file_path = 'CSR_GRACE_GRACE-FO_RL0603_Mascons_all-corrections.nc'
ds = xr.open_dataset(file_path)
fields = list(ds.variables.keys())
print(fields)


['time_bounds', 'lwe_thickness', 'time', 'lon', 'lat']


In [4]:
ds = ds.assign_coords(
    time=pd.to_datetime(ds['time'].values, unit='D', origin='2002-01-01')
)
print(ds['time'].dtype)        # sanity check: should be datetime64[ns]
print(ds['time'].values[:3])   # sanity check: should print real dates


datetime64[ns]
['2002-04-18T00:00:00.000000000' '2002-05-10T12:00:00.000000000'
 '2002-08-16T12:00:00.000000000']


In [5]:
R = 6378136.3  # earth radius in meters (from user_note_3)
res_deg = 0.25 # grid resolution in degrees
d_lat = np.radians(res_deg)
d_lon = np.radians(res_deg)

grid_areas = (R ** 2) * np.cos(np.radians(ds.lat)) * d_lat * d_lon
rho_water = 1000  # kg/m^3 -- freshwater; was 1025 (seawater) in original, corrected

lwe_meters = ds['lwe_thickness'] / 100.0  # cm -> m

ds['mass_anomaly'] = lwe_meters * grid_areas * rho_water
ds['mass_anomaly'].attrs['units'] = 'kg'
ds['mass_anomaly'].attrs['long_name'] = 'Total mass anomaly per grid cell'


In [ ]:
twsa = ds['lwe_thickness']

twsa_shifted = twsa.assign_coords(lon=(((twsa.lon + 180) % 360) - 180)).sortby('lon')

region = twsa_shifted.sel(lat=slice(-35, 38), lon=slice(-18, 52))
clim = region.groupby('time.month').mean('time')
anom = region.groupby('time.month') - clim  

In [ ]:
flat_data = anom.stack(gridcell=('lat', 'lon')).dropna('gridcell')
print(region.shape, anom.shape, flat_data.sizes)

import time

print("Calculating correlation matrix... ", end="", flush=True)
start_time = time.time()

matrix = flat_data.values.T
corr_matrix = np.corrcoef(matrix).astype(np.float32)
np.fill_diagonal(corr_matrix, 0.0)
corr_matrix = np.nan_to_num(corr_matrix, nan=0.0, posinf=0.0, neginf=0.0)

end_time = time.time()


(257, 292, 280) (257, 292, 280) Frozen({'time': 257, 'gridcell': 81760})
Calculating correlation matrix... 

In [ ]:
threshold = 0.95
mask = corr_matrix >= threshold
np.fill_diagonal(mask, False)

rows, cols = np.where(mask)
values = corr_matrix[rows, cols]
sparse_corr = sp.csr_matrix((values, (rows, cols)), shape=corr_matrix.shape)

print("edges:", sparse_corr.nnz // 2)
# check this number before proceeding -- if it's still tens of millions,
# raise `threshold` or add spatial-neighbor restriction before running Louvain

# free the big dense arrays now that we have the sparse version
del corr_matrix, mask
gc.collect()


In [ ]:
G = nx.from_scipy_sparse_array(sparse_corr)
communities = nx.community.louvain_communities(G, weight='weight')


In [ ]:
n_nodes = sparse_corr.shape[0]
basin_mask = np.zeros(n_nodes)

for basin_id, community in enumerate(communities):
    for node in community:
        basin_mask[node] = basin_id

basin_flat = xr.DataArray(
    basin_mask,
    dims=['gridcell'],
    coords={'gridcell': flat_data.gridcell}
)
basin_da = basin_flat.unstack('gridcell').astype(float)

print(basin_da)
print("\nDimensions of basin_da:", basin_da.shape)
print("Total number of data points:", basin_da.size)
print("Total number of valid numbers:", basin_da.notnull().sum().values)


In [ ]:
plt.figure(figsize=(12, 8))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.add_feature(cfeature.COASTLINE, linestyle='-', linewidth=1, edgecolor='black')
ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5, edgecolor='gray')
ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=0)

ax.set_global()
im = basin_da.plot(
    ax=ax,
    x='lon',
    y='lat',
    transform=ccrs.PlateCarree(),
    cmap='tab20',
    add_colorbar=True,
    cbar_kwargs={'label': 'Custom Basin ID', 'shrink': 0.6}
)
# ax.set_extent(
#     [region.lon.min(), region.lon.max(), region.lat.min(), region.lat.max()],
#     crs=ccrs.PlateCarree()
# )

plt.title('Custom Hydrological Basins from GRACE Connectivity', fontsize=14, fontweight='bold')
plt.show()